In [2]:
print(agents["registered_on"].dtype)
print(agents["registered_on"].head())

object
0    2025-04-09
1    2025-11-14
2    2024-10-13
3    2024-10-09
4    2025-01-18
Name: registered_on, dtype: object


In [5]:
# connect and pull the tables
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

agents = pd.read_sql("SELECT * FROM agents", engine)
transactions = pd.read_sql("SELECT * FROM transactions", engine)

# make sure these come through as real dates, not text
agents["registered_on"] = pd.to_datetime(agents["registered_on"])
transactions["txn_ts"] = pd.to_datetime(transactions["txn_ts"])


# cohort month per agent, same as DATE_TRUNC('month', registered_on) in sql
agents["cohort_month"] = agents["registered_on"].dt.to_period("M").dt.to_timestamp()

# how many agents per cohort
cohort_sizes = agents.groupby("cohort_month").size().reset_index(name="cohort_size")


# attach each agent's cohort month onto their transactions
# validate=many_to_one checks each txn matches exactly one agent, same
# assumption the sql join was relying on
txn_with_cohort = transactions.merge(
    agents[["agent_id", "cohort_month"]],
    on="agent_id",
    how="inner",
    validate="many_to_one",
)

txn_with_cohort["activity_month"] = txn_with_cohort["txn_ts"].dt.to_period("M").dt.to_timestamp()


# one row per agent per month they were active, duplicates dropped
# same as the DISTINCT in the sql version
activity = txn_with_cohort[["agent_id", "cohort_month", "activity_month"]].drop_duplicates()


# months since registration, same math as the sql EXTRACT/month_number calc
activity["month_number"] = (
    (activity["activity_month"].dt.year - activity["cohort_month"].dt.year) * 12
    + (activity["activity_month"].dt.month - activity["cohort_month"].dt.month)
)

activity_filtered = activity[activity["month_number"].between(1, 6)]


# count distinct active agents per cohort per month offset
retained = (
    activity_filtered.groupby(["cohort_month", "month_number"])["agent_id"]
    .nunique()
    .reset_index(name="active_agents")
)

pandas_retention = retained.merge(cohort_sizes, on="cohort_month")
pandas_retention["retention_rate"] = (
    pandas_retention["active_agents"] / pandas_retention["cohort_size"]
).round(4)
pandas_retention = pandas_retention.sort_values(
    ["cohort_month", "month_number"]
).reset_index(drop=True)


# same query as sql/analytics.sql Q1, run here so we can diff against it
sql_retention = pd.read_sql("""
    WITH cohorts AS (
        SELECT
            agent_id,
            DATE_TRUNC('month', registered_on) AS cohort_month
        FROM agents
    ),
    cohort_sizes AS (
        SELECT
            cohort_month,
            COUNT(*) AS cohort_size
        FROM cohorts
        GROUP BY cohort_month
    ),
    agent_activity_months AS (
        SELECT DISTINCT
            c.agent_id,
            c.cohort_month,
            DATE_TRUNC('month', t.txn_ts) AS activity_month
        FROM cohorts c
        JOIN transactions t ON t.agent_id = c.agent_id
    ),
    month_offsets AS (
        SELECT
            agent_id,
            cohort_month,
            (EXTRACT(YEAR FROM activity_month) - EXTRACT(YEAR FROM cohort_month)) * 12
              + (EXTRACT(MONTH FROM activity_month) - EXTRACT(MONTH FROM cohort_month)) AS month_number
        FROM agent_activity_months
    ),
    retained AS (
        SELECT
            cohort_month,
            month_number,
            COUNT(DISTINCT agent_id) AS active_agents
        FROM month_offsets
        WHERE month_number BETWEEN 1 AND 6
        GROUP BY cohort_month, month_number
    )
    SELECT
        r.cohort_month,
        r.month_number,
        r.active_agents,
        s.cohort_size,
        ROUND(r.active_agents::numeric / s.cohort_size, 4) AS retention_rate
    FROM retained r
    JOIN cohort_sizes s ON s.cohort_month = r.cohort_month
    ORDER BY r.cohort_month, r.month_number
""", engine)

# drop timezone so this matches the pandas-built cohort_month column
sql_retention["cohort_month"] = sql_retention["cohort_month"].dt.tz_localize(None)


# line up both results and check they agree
compare = pandas_retention.merge(
    sql_retention,
    on=["cohort_month", "month_number"],
    suffixes=("_py", "_sql"),
)
compare["retention_diff"] = (
    compare["retention_rate_py"].astype(float) - compare["retention_rate_sql"].astype(float)
)

mismatches = compare[compare["retention_diff"].abs() > 0.0001]

print("Rows compared:", len(compare))
print("Mismatched rows:", len(mismatches))
print(mismatches)

Rows compared: 0
Mismatched rows: 0
Empty DataFrame
Columns: [cohort_month, month_number, active_agents_py, cohort_size_py, retention_rate_py, active_agents_sql, cohort_size_sql, retention_rate_sql, retention_diff]
Index: []


In [11]:
# connect and load tables
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

agents = pd.read_sql("SELECT * FROM agents", engine)
transactions = pd.read_sql("SELECT * FROM transactions", engine)
services = pd.read_sql("SELECT * FROM services", engine)

agents["registered_on"] = pd.to_datetime(agents["registered_on"])
transactions["txn_ts"] = pd.to_datetime(transactions["txn_ts"])


# build the joined table, validate= checks the merge is really the
# relationship we think it is (each txn matches exactly one agent, etc)
joined = transactions.merge(
    agents,
    on="agent_id",
    how="left",
    validate="many_to_one",
)

joined = joined.merge(
    services,
    on="service_code",
    how="left",
    validate="many_to_one",
)


# per-agent rank - rank each agent by their total transaction value,
# same rank applied to every row belonging to that agent
agent_totals = joined.groupby("agent_id")["amount_xaf"].transform("sum")
joined["agent_total_value"] = agent_totals
joined["agent_rank"] = joined["agent_total_value"].rank(method="dense", ascending=False).astype(int)


# per-division share - each agent's total value as a share of their division's total
division_totals = joined.groupby("division")["amount_xaf"].transform("sum")
joined["division_share"] = (agent_totals / division_totals).round(4)


# pivot: total amount per service category per month
joined["txn_month"] = joined["txn_ts"].dt.to_period("M").dt.to_timestamp()

pivoted = joined.pivot_table(
    index="txn_month",
    columns="category",
    values="amount_xaf",
    aggfunc="sum",
    fill_value=0,
)

# melt back to long format
melted = pivoted.reset_index().melt(
    id_vars="txn_month",
    var_name="category",
    value_name="amount_xaf",
)


# check the pivot/melt round trip didn't lose anything - total amount
# before and after should match exactly
original_total = joined.groupby(["txn_month", "category"])["amount_xaf"].sum().reset_index()

check = original_total.merge(
    melted,
    on=["txn_month", "category"],
    suffixes=("_original", "_melted"),
)
check["diff"] = check["amount_xaf_original"] - check["amount_xaf_melted"]

print("Rows checked:", len(check))
print("Mismatched rows:", len(check[check["diff"] != 0]))
print(check[check["diff"] != 0])

Rows checked: 180
Mismatched rows: 0
Empty DataFrame
Columns: [txn_month, category, amount_xaf_original, amount_xaf_melted, diff]
Index: []


In [12]:
joined[["agent_id", "agent_total_value", "agent_rank", "division", "division_share"]].head(10)

,agent_id,agent_total_value,agent_rank,division,division_share
0,AG5591,57133337.0,5,Menchum,0.1211
1,AG5802,7643188.0,437,Mezam,0.0028
2,AG5232,20850140.0,162,Ngo-Ketunjia,0.0143
3,AG5592,12350982.0,301,Mezam,0.0045
4,AG6103,39114572.0,42,Ngo-Ketunjia,0.0268
5,AG6059,8991869.0,389,Mezam,0.0033
6,AG5640,29864508.0,97,Boyo,0.0199
7,AG5971,41802335.0,33,Boyo,0.0278
8,AG6154,37248783.0,51,Bui,0.0312
9,AG6138,9020898.0,385,Momo,0.0066


In [13]:
# memory usage before any changes
print("before:")
joined.info(memory_usage="deep")


# work on a copy so we can compare before vs after side by side
optimized = joined.copy()


# repeated text columns - category dtype stores each unique value once
# instead of repeating the same string in every row
category_columns = [
    "agent_id", "service_code", "status_x", "status_y",
    "town", "division", "agent_type", "service_name", "category",
]
for col in category_columns:
    if col in optimized.columns:
        optimized[col] = optimized[col].astype("category")


# downcast numbers to the smallest type that still fits the values
float_columns = [
    "amount_xaf", "fee_xaf", "float_limit_xaf", "base_fee_pct",
    "agent_total_value", "division_share",
]
for col in float_columns:
    if col in optimized.columns:
        optimized[col] = pd.to_numeric(optimized[col], downcast="float")

int_columns = ["agent_rank"]
for col in int_columns:
    if col in optimized.columns:
        optimized[col] = pd.to_numeric(optimized[col], downcast="integer")


# memory usage after
print("after:")
optimized.info(memory_usage="deep")


# work out the actual percentage saved
before_bytes = joined.memory_usage(deep=True).sum()
after_bytes = optimized.memory_usage(deep=True).sum()
reduction = 1 - (after_bytes / before_bytes)

print(f"before: {before_bytes:,} bytes")
print(f"after: {after_bytes:,} bytes")
print(f"reduction: {reduction:.2%}")

before:
<class 'pandas.DataFrame'>
RangeIndex: 160285 entries, 0 to 160284
Data columns (total 22 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   txn_id             160285 non-null  str           
 1   agent_id           160285 non-null  str           
 2   service_code       160285 non-null  str           
 3   txn_ts             160285 non-null  datetime64[us]
 4   amount_xaf         160285 non-null  float64       
 5   fee_xaf            160285 non-null  float64       
 6   status_x           160285 non-null  str           
 7   customer_msisdn    160285 non-null  str           
 8   agent_name         160285 non-null  str           
 9   town               160285 non-null  str           
 10  division           160285 non-null  str           
 11  registered_on      160285 non-null  datetime64[s] 
 12  agent_type         160285 non-null  str           
 13  float_limit_xaf    160285 non-null  float64    